In [1]:
%use dataframe
%use kandy
import profilelib.*

In [2]:
val profileData = load_data("jfrs/serialization/cstack_dwarf/") { it
    .filterNot { it.any { it.method.name.contains("::Sweep<") } }
    .filterNot { it.any { it.method.name.contains("JsonToStringWriter") } }
}

'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=43,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=native,cycles=100000,sampleInterval=100,iteration=10.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=44,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=50,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=20,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/data/jfrs/serialization/cstack_dwarf/platform=jvm,cycles=100000,sampleInterval=100,iteration=48,warmup=3.jfr'
'/home/martinzimen/IdeaProjects/stdlib-profiling/dat

In [3]:
fun getResult(name: String, jvmName: String, nativeName: String): Pair<String, Double> =
    profileData.funRatio(jvmName, nativeName)
    .map { it!! }
    .also { println(name + ": " + formattedMeanCV(it)) }
    .let { name to mean(it) }
val profileResults = mapOf(
    getResult("Map", "java.util.HashMap.get", "kotlin.collections.Map.get"),
    getResult("StringBuilder", "java.lang.StringBuilder.append", "kotlin.text.StringBuilder.append"),
    getResult("StringSubstring", "java.lang.String.substring", "weirdo:Kotlin_String_subSequence"),
)

Map: 0.129 (0.032)
StringBuilder: 0.154 (0.036)
StringSubstring: 0.174 (0.050)


In [4]:
@DataSchema
data class BenchmarkRow(
    val benchmark: String,
    val jvm: Double,
    val jvmError: Double,
    val native: Double,
    val nativeError: Double
)

fun loadAsDF(path: String): DataFrame<BenchmarkRow> {
    val jvm = loadScores(loadJson(dataPath(path.replace("*", "jvm"))))
    val native = loadScores(loadJson(dataPath(path.replace("*", "linuxX64"))))
    if (jvm.keys != native.keys) throw Exception()
    val benchmarks = jvm.keys.toList()

    fun Map<String, Double>.toNamedColumn(name: String): DataColumn<Double> = benchmarks
        .map { this[it]!! }
        .toColumn(name)

    return dataFrameOf(
        benchmarks.toColumn("benchmark"),
        jvm.mapValues { it.value.first }.toNamedColumn("jvm"),
        jvm.mapValues { it.value.second }.toNamedColumn("jvmError"),
        native.mapValues { it.value.first }.toNamedColumn("native"),
        native.mapValues { it.value.second }.toNamedColumn("nativeError"),
    ) as DataFrame<BenchmarkRow>
}

In [10]:
fun findControlName(bmName: String): String = when (bmName.substringBefore(".")) {
    "MapBenchmark" -> bmName
                        .replace(".konst{", ".konst_control{")
                        .replace(".random{", ".random_control{")
    "StringSubstringBenchmark" -> bmName.substringBefore("_") + "_control"
    "StringBuilderBenchmark" -> "StringBuilderBenchmark.control"
    else -> throw Exception()
}

val benchmarkDF = loadAsDF("benchmarks/report-selection-*.json")
    .filter { !benchmark.contains("ConstMapBenchmark") }
    .filter { !benchmark.contains("StringSubstringBenchmark.big") }
    .filter { !benchmark.endsWith("_none") }
    .filter { !benchmark.endsWith("_all") }
    .add("jvmControl") { df().single{ benchmark == findControlName(this@add.benchmark) }.jvm }
    .add("jvmControlError") { df().single{ benchmark == findControlName(this@add.benchmark) }.jvmError }
    .add("nativeControl") { df().single{ benchmark == findControlName(this@add.benchmark) }.native }
    .add("nativeControlError") { df().single{ benchmark == findControlName(this@add.benchmark) }.nativeError }
    .filter { !benchmark.contains("control") }

In [15]:
val percGCDF = dataFrameOf("benchmark", "jvm", "native")(
    "StringSubstringBenchmark.small_some", 0.012, 0.159,
    "StringSubstringBenchmark.medium_some", 0.012, 0.15
)
percGCDF

benchmark,jvm,native
StringSubstringBenchmark.small_some,0.012000,0.159000
StringSubstringBenchmark.medium_some,0.012000,0.150000


In [25]:
val benchmarkResults = benchmarkDF
    .add("ratio") { (jvm - jvmControl) / (native - nativeControl) }
    .add("ratioError") { propagateErrorThroughRatio(
        jvm - jvmControl,
        propagateErrorThroughSum(jvmError, jvmControlError),
        native - nativeControl,
        propagateErrorThroughSum(nativeError, nativeControlError),
    )}
    .add("ratiowGC") {
        percGCDF.singleOrNull{ this@add.benchmark == this@singleOrNull.benchmark }?.let { percGC ->
            (
                (1 - percGC.jvm) * (jvm - jvmControl)
            ) / (
                (1 - percGC.native) * (native - nativeControl)
            )
        } ?: this["ratio"] as Double
    }

In [26]:
benchmarkResults.select { benchmark and ratio and ratioError and ratiowGC }

benchmark,ratio,ratioError,ratiowGC
"MapBenchmark.konst{miss=""true"",size=""...",0.044342,0.000002,0.044342
"MapBenchmark.konst{miss=""true"",size=""...",0.051058,0.000002,0.051058
"MapBenchmark.konst{miss=""true"",size=""...",0.053358,0.000001,0.053358
"MapBenchmark.konst{miss=""true"",size=""...",0.052532,0.000000,0.052532
"MapBenchmark.konst{miss=""false"",size=...",0.048173,0.000002,0.048173
"MapBenchmark.konst{miss=""false"",size=...",0.047178,0.000003,0.047178
"MapBenchmark.konst{miss=""false"",size=...",0.054072,0.000001,0.054072
"MapBenchmark.konst{miss=""false"",size=...",0.054862,0.000001,0.054862
"MapBenchmark.random{miss=""true"",size=...",0.075438,0.000004,0.075438
"MapBenchmark.random{miss=""true"",size=...",0.080866,0.000003,0.080866


In [28]:
plot {
    points {
        x(benchmarkResults.benchmark.toList().map { it.substringBefore("Benchmark") })
        y(benchmarkResults.ratiowGC.toList())
    }
    points {
        x(profileResults.keys)
        y(profileResults.values)
        symbol = Symbol.CROSS
        size = 10.0
    }
}

<head>
 <meta charset="UTF-8">
 <style> html, body { margin: 0; padding: 0; overflow: hidden; } </style>
 <script type="text/javascript" data-lets-plot-script="library" src="https://cdn.jsdelivr.net/gh/JetBrains/lets-plot@v4.5.1/js-package/distr/lets-plot.min.js"></script>
 </head>
 <body>
 <div id="10AZbx"></div>
 <script type="text/javascript" data-lets-plot-script="plot">
 
 (function() {
 // ----------
 
 const forceImmediateRender = false;
 const responsive = false;
 
 let sizing = {
 width_mode: "FIXED",
 height_mode: "FIXED",
 width: 600.0, 
 height: 400.0 
 };
 
 const preferredWidth = document.body.dataset.letsPlotPreferredWidth;
 if (preferredWidth !== undefined) {
 sizing = {
 width_mode: 'FIXED',
 height_mode: 'SCALED',
 width: parseFloat(preferredWidth)
 };
 }
 
 const containerDiv = document.getElementById("10AZbx");
 let fig = null;
 
 function renderPlot() {
 if (fig === null) {
 const plotSpec = {
"mapping":{
},
"data":{
},
"kind":"plot",
"scales":[{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
},{
"aesthetic":"x",
"discrete":true
},{
"aesthetic":"y",
"limits":[null,null]
}],
"layers":[{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["Map","Map","Map","Map","Map","Map","Map","Map","Map","Map","Map","Map","Map","Map","Map","Map","StringBuilder","StringBuilder","StringBuilder","StringSubstring","StringSubstring"],
"y":[0.04434194127005931,0.05105787823865348,0.05335782174979124,0.05253219758253958,0.0481729983475538,0.04717834643981149,0.054071604786462155,0.05486213852229973,0.0754376836639799,0.08086555885555434,0.09376901752732111,0.25586737070457594,0.0635732560676073,0.0781273561356847,0.06821422480042891,0.13815493726327555,0.11654591437776315,0.15908913855310286,0.10286007839542344,0.20023067910150508,0.19293827666523822]
},
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
},{
"mapping":{
"x":"x",
"y":"y"
},
"stat":"identity",
"data":{
"x":["Map","StringBuilder","StringSubstring"],
"y":[0.12880429621964468,0.15396163188536277,0.17360175475443818]
},
"shape":4.0,
"size":10.0,
"sampling":"none",
"inherit_aes":false,
"position":"identity",
"geom":"point",
"data_meta":{
"series_annotations":[{
"type":"str",
"column":"x"
},{
"type":"float",
"column":"y"
}]
}
}],
"spec_id":"17"
};
 fig = LetsPlot.buildPlotFromProcessedSpecs(plotSpec, containerDiv, sizing);
 } else {
 fig.updateView({});
 }
 }
 
 const renderImmediately = 
 forceImmediateRender || (
 sizing.width_mode === 'FIXED' && 
 (sizing.height_mode === 'FIXED' || sizing.height_mode === 'SCALED')
 );
 
 if (renderImmediately) {
 renderPlot();
 }
 
 if (!renderImmediately || responsive) {
 // Set up observer for initial sizing or continuous monitoring
 var observer = new ResizeObserver(function(entries) {
 for (let entry of entries) {
 if (entry.contentBoxSize && 
 entry.contentBoxSize[0].inlineSize > 0) {
 if (!responsive && observer) {
 observer.disconnect();
 observer = null;
 }
 renderPlot();
 if (!responsive) {
 break;
 }
 }
 }
 });
 
 observer.observe(containerDiv);
 }
 
 // ----------
 })();
 
 </script>
 </body>
</html>"> 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 Map 
 
 
 
 
 
 
 
 
 StringBuilder 
 
 
 
 
 
 
 
 
 StringSubstring 
 
 
 
 
 
 
 
 
 
 
 0.05 
 
 
 
 
 
 
 0.1 
 
 
 
 
 
 
 0.15 
 
 
 
 
 
 
 0.2 
 
 
 
 
 
 
 0.25 
 
 
 
 
 
 
 
 
 y 
 
 
 
 
 x